<h2> Adding Fire Frequency to the GeoDataFrame of All Pairs <h2>

In [1]:
import os
import glob
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from scipy.stats import mode
from shapely.geometry import mapping
from tqdm import tqdm
from rasterio.io import MemoryFile
from shapely import wkt
from shapely.geometry import box
from pyproj import CRS

<h5> Loading the Pairs <h5>


The "Pairs" GeoPackage (.gpkg) is the final product of Notebooks 0 , 1 and 2. 

In [2]:
#Load the pairs
pairs_path = r"GEDI_Pairs/GEDI_Footprint_Fire_Pairs.gpkg"
pairs = gpd.read_file(pairs_path)

print("Layer CRS (geometry):", pairs.crs)  # should be EPSG:4326
pairs.head()

Layer CRS (geometry): EPSG:4326


,index_1,index_2,distance_m,fire,fire_date,post_fire_months,geometry_1,geometry_2,latitude_1,longitude_1,...,fire_count_2,first_fire_date_2,last_fire_date_2,fire_dates_str_2,delta_agbd,source_file,pairing_crs,geometry_crs_1,geometry_crs_2,geometry
0,15746,15747,3.030845,True,2020-07-01 00:00:00,21.0,"POLYGON ((-51.045541 -4.228863, -51.045544 -4....","POLYGON ((-51.045564 -4.228848, -51.045567 -4....",-4.228843,-51.045745,...,2,2020-07-01 00:00:00,2022-08-01 00:00:00,"2020-07-01,2022-08-01",15.451023,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-51.04574 -4.22884, -51.04577 -4.2..."
1,7437,7438,3.827840,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.643081 -4.824114, -49.643084 -4....","POLYGON ((-49.643071 -4.824081, -49.643074 -4....",-4.824094,-49.643285,...,1,2022-09-01 00:00:00,2022-09-01 00:00:00,2022-09-01,54.691750,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64328 -4.82409, -49.64327 -4.8..."
2,7439,7440,3.939691,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.64338 -4.823694, -49.643382 -4.8...","POLYGON ((-49.643369 -4.82366, -49.643372 -4.8...",-4.823674,-49.643583,...,1,2022-09-01 00:00:00,2022-09-01 00:00:00,2022-09-01,3.437462,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64358 -4.82367, -49.64357 -4.8..."
3,32267,32268,4.106089,True,2020-10-01 00:00:00,11.0,"POLYGON ((-50.521602 -3.564744, -50.521605 -3....","POLYGON ((-50.521572 -3.564723, -50.521575 -3....",-3.564724,-50.521806,...,2,2020-10-01 00:00:00,2022-09-01 00:00:00,"2020-10-01,2022-09-01",-178.882935,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-50.52181 -3.56472, -50.52178 -3.5..."
4,9379,9378,4.224179,True,2020-11-01 00:00:00,21.0,"POLYGON ((-49.662477 -4.462845, -49.66248 -4.4...","POLYGON ((-49.66251 -4.462863, -49.662513 -4.4...",-4.462825,-49.662681,...,2,2020-11-01 00:00:00,2022-10-01 00:00:00,"2020-11-01,2022-10-01",307.078278,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.66268 -4.46282, -49.66271 -4.4..."


In [3]:
#Load Fire Frequency raster 
#Comes from Mapbiomas Collection 4 from 1985 - 2024

#Load the rasters
ff_path = r"Fire History/Mapbioma_Collection_4_Fire_Frequency_AoD_1985_2024.tif" 

ff_ds = rasterio.open(ff_path)

In [4]:
##### USE THIS PART WHENEVER USING THE ORIGINAL GEDI_Footprint_Pairs_Fire.gpkg

# Parse WKT → shapely geometries 

# When pre_geom and post_geom are WKT in EPSG:4326 - IN LAT/LONG
pre_ll  = gpd.GeoSeries(pairs["geometry_1"].apply(wkt.loads),  crs="EPSG:4326")
post_ll = gpd.GeoSeries(pairs["geometry_2"].apply(wkt.loads), crs="EPSG:4326")


# Assign back to the DataFrame
pairs["pre_geom"]  = pre_ll
pairs["post_geom"] = post_ll


# Do NOT touch pairs.crs or pairs.geometry
print("Layer CRS after:", pairs.crs)  # still EPSG:4326
print("Example geometry:", pairs.loc[0, "geometry"])
print("Example pre_geom:", pairs.loc[0, "geometry_1"])
print("Example post_geom:", pairs.loc[0, "geometry_2"])
pairs.head()


Layer CRS after: EPSG:4326
Example geometry: LINESTRING (-51.04574484879311 -4.228842861565982, -51.04576788861222 -4.228828138089475)
Example pre_geom: POLYGON ((-51.045541 -4.228863, -51.045544 -4.228883, -51.045549 -4.228902, -51.045556 -4.228921, -51.045564 -4.22894, -51.045575 -4.228957, -51.045587 -4.228973, -51.0456 -4.228988, -51.045615 -4.229002, -51.045631 -4.229014, -51.045648 -4.229024, -51.045667 -4.229033, -51.045685 -4.229039, -51.045705 -4.229044, -51.045725 -4.229047, -51.045745 -4.229048, -51.045765 -4.229047, -51.045785 -4.229044, -51.045804 -4.229039, -51.045823 -4.229033, -51.045841 -4.229024, -51.045859 -4.229014, -51.045875 -4.229002, -51.045889 -4.228988, -51.045903 -4.228973, -51.045915 -4.228957, -51.045925 -4.22894, -51.045934 -4.228921, -51.045941 -4.228902, -51.045945 -4.228883, -51.045948 -4.228863, -51.045949 -4.228843, -51.045948 -4.228823, -51.045945 -4.228803, -51.045941 -4.228783, -51.045934 -4.228764, -51.045925 -4.228746, -51.045915 -4.228729, -51.0

,index_1,index_2,distance_m,fire,fire_date,post_fire_months,geometry_1,geometry_2,latitude_1,longitude_1,...,last_fire_date_2,fire_dates_str_2,delta_agbd,source_file,pairing_crs,geometry_crs_1,geometry_crs_2,geometry,pre_geom,post_geom
0,15746,15747,3.030845,True,2020-07-01 00:00:00,21.0,"POLYGON ((-51.045541 -4.228863, -51.045544 -4....","POLYGON ((-51.045564 -4.228848, -51.045567 -4....",-4.228843,-51.045745,...,2022-08-01 00:00:00,"2020-07-01,2022-08-01",15.451023,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-51.04574 -4.22884, -51.04577 -4.2...","POLYGON ((-51.04554 -4.22886, -51.04554 -4.228...","POLYGON ((-51.04556 -4.22885, -51.04557 -4.228..."
1,7437,7438,3.827840,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.643081 -4.824114, -49.643084 -4....","POLYGON ((-49.643071 -4.824081, -49.643074 -4....",-4.824094,-49.643285,...,2022-09-01 00:00:00,2022-09-01,54.691750,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64328 -4.82409, -49.64327 -4.8...","POLYGON ((-49.64308 -4.82411, -49.64308 -4.824...","POLYGON ((-49.64307 -4.82408, -49.64307 -4.824..."
2,7439,7440,3.939691,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.64338 -4.823694, -49.643382 -4.8...","POLYGON ((-49.643369 -4.82366, -49.643372 -4.8...",-4.823674,-49.643583,...,2022-09-01 00:00:00,2022-09-01,3.437462,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64358 -4.82367, -49.64357 -4.8...","POLYGON ((-49.64338 -4.82369, -49.64338 -4.823...","POLYGON ((-49.64337 -4.82366, -49.64337 -4.823..."
3,32267,32268,4.106089,True,2020-10-01 00:00:00,11.0,"POLYGON ((-50.521602 -3.564744, -50.521605 -3....","POLYGON ((-50.521572 -3.564723, -50.521575 -3....",-3.564724,-50.521806,...,2022-09-01 00:00:00,"2020-10-01,2022-09-01",-178.882935,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-50.52181 -3.56472, -50.52178 -3.5...","POLYGON ((-50.5216 -3.56474, -50.5216 -3.56476...","POLYGON ((-50.52157 -3.56472, -50.52158 -3.564..."
4,9379,9378,4.224179,True,2020-11-01 00:00:00,21.0,"POLYGON ((-49.662477 -4.462845, -49.66248 -4.4...","POLYGON ((-49.66251 -4.462863, -49.662513 -4.4...",-4.462825,-49.662681,...,2022-10-01 00:00:00,"2020-11-01,2022-10-01",307.078278,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.66268 -4.46282, -49.66271 -4.4...","POLYGON ((-49.66248 -4.46284, -49.66248 -4.462...","POLYGON ((-49.66251 -4.46286, -49.66251 -4.462..."


In [5]:
# Create a buffer (meters), but keep stored geometries in EPSG:4326

buffer_size = 20  # meters

pre_col_name = f"pre_buff_{buffer_size}"
post_col_name = f"post_buff_{buffer_size}"

# Create columns
pairs[pre_col_name] = None
pairs[post_col_name] = None

# Buffer by pairing CRS (important because rows can be in different UTM zones)
# Using different zonal projections, which is different for each pair. 

for pcrs, idxs in pairs.groupby("pairing_crs").groups.items():
    if pd.isna(pcrs):
        print(f"Skipping rows with missing pairing_crs: {len(idxs)}")
        continue

    # pre/post footprint polygons are already EPSG:4326 from the previous cell
    pre_ll_group = gpd.GeoSeries(pairs.loc[idxs, "pre_geom"], crs="EPSG:4326")
    post_ll_group = gpd.GeoSeries(pairs.loc[idxs, "post_geom"], crs="EPSG:4326")

    # project to meter-based CRS, buffer, then return to EPSG:4326
    pre_buff_ll = pre_ll_group.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326")
    post_buff_ll = post_ll_group.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326")

    pairs.loc[idxs, pre_col_name] = pre_buff_ll.values
    pairs.loc[idxs, post_col_name] = post_buff_ll.values

print(f"Created columns: {pre_col_name}, {post_col_name}")
pairs.head()



Created columns: pre_buff_20, post_buff_20


,index_1,index_2,distance_m,fire,fire_date,post_fire_months,geometry_1,geometry_2,latitude_1,longitude_1,...,delta_agbd,source_file,pairing_crs,geometry_crs_1,geometry_crs_2,geometry,pre_geom,post_geom,pre_buff_20,post_buff_20
0,15746,15747,3.030845,True,2020-07-01 00:00:00,21.0,"POLYGON ((-51.045541 -4.228863, -51.045544 -4....","POLYGON ((-51.045564 -4.228848, -51.045567 -4....",-4.228843,-51.045745,...,15.451023,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-51.04574 -4.22884, -51.04577 -4.2...","POLYGON ((-51.04554 -4.22886, -51.04554 -4.228...","POLYGON ((-51.04556 -4.22885, -51.04557 -4.228...",POLYGON ((-51.04536100435041 -4.22887207121828...,POLYGON ((-51.045384004353934 -4.2288570712184...
1,7437,7438,3.827840,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.643081 -4.824114, -49.643084 -4....","POLYGON ((-49.643071 -4.824081, -49.643074 -4....",-4.824094,-49.643285,...,54.691750,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64328 -4.82409, -49.64327 -4.8...","POLYGON ((-49.64308 -4.82411, -49.64308 -4.824...","POLYGON ((-49.64307 -4.82408, -49.64307 -4.824...",POLYGON ((-49.64290090729956 -4.82412306098943...,POLYGON ((-49.642890907309024 -4.8240900609897...
2,7439,7440,3.939691,True,2022-09-01 00:00:00,4.0,"POLYGON ((-49.64338 -4.823694, -49.643382 -4.8...","POLYGON ((-49.643369 -4.82366, -49.643372 -4.8...",-4.823674,-49.643583,...,3.437462,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.64358 -4.82367, -49.64357 -4.8...","POLYGON ((-49.64338 -4.82369, -49.64338 -4.823...","POLYGON ((-49.64337 -4.82366, -49.64337 -4.823...",POLYGON ((-49.64319990738833 -4.82370306099626...,POLYGON ((-49.643188907398134 -4.8236690609966...
3,32267,32268,4.106089,True,2020-10-01 00:00:00,11.0,"POLYGON ((-50.521602 -3.564744, -50.521605 -3....","POLYGON ((-50.521572 -3.564723, -50.521575 -3....",-3.564724,-50.521806,...,-178.882935,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-50.52181 -3.56472, -50.52178 -3.5...","POLYGON ((-50.5216 -3.56474, -50.5216 -3.56476...","POLYGON ((-50.52157 -3.56472, -50.52158 -3.564...",POLYGON ((-50.52142215206022 -3.56475307829227...,POLYGON ((-50.52139215206509 -3.56473207829234...
4,9379,9378,4.224179,True,2020-11-01 00:00:00,21.0,"POLYGON ((-49.662477 -4.462845, -49.66248 -4.4...","POLYGON ((-49.66251 -4.462863, -49.662513 -4.4...",-4.462825,-49.662681,...,307.078278,pairs_GEDI_Fire_Dates_0000000000-0000026880_ba...,EPSG:32722,EPSG:4326,EPSG:4326,"LINESTRING (-49.66268 -4.46282, -49.66271 -4.4...","POLYGON ((-49.66248 -4.46284, -49.66248 -4.462...","POLYGON ((-49.66251 -4.46286, -49.66251 -4.462...",POLYGON ((-49.66229699774702 -4.46285406584504...,POLYGON ((-49.662329997740194 -4.4628720658448...


In [6]:
print("pairs CRS:", pairs.crs) 
print("ff_ds CRS:", ff_ds.crs)

pairs CRS: EPSG:4326
ff_ds CRS: EPSG:4326


In [7]:
#Creating the function to extract Fire Frequency (from raster) of each small polygon 

def extract_info_from_polygon(geom, ds):
    try:
        out_img, _ = mask(ds, [mapping(geom)], crop=True, filled=False, all_touched=True)
    except Exception:
        return None

    data = out_img[0].flatten()

    # Remove nodata
    nodata = ds.nodata
    if nodata is not None:
        data = data[data != nodata]

    # Remove NaN
    data = data[~np.isnan(data)]

    if len(data) == 0:
        return None

    # Return the highest fire frequency pixel
    return int(np.max(data))


#Function to covert column to wkt (not an active geomtry, just to store the information)
def to_wkt_column(df, col): 
    # Make sure everything is geometry 
    df[col] = df[col].apply( 
        lambda x: wkt.loads(x) if isinstance(x, str) else x ) 
    # Convert geometry to WKT 
    df[col] = gpd.GeoSeries(df[col]).to_wkt()

In [8]:
#Appplying the function 

pre_ff_list = []
post_ff_list = []

for idx, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Extracting Fire Frequency"):
    pre_geom = row[f"pre_buff_{buffer_size}"] 
    post_geom = row[f"post_buff_{buffer_size}"]

    pre_ff = extract_info_from_polygon(pre_geom, ff_ds)
    post_ff = extract_info_from_polygon(post_geom, ff_ds)

    pre_ff_list.append(pre_ff)
    post_ff_list.append(post_ff)

pairs["pre_FF"] = pre_ff_list
pairs["post_FF"] = post_ff_list

Extracting Fire Frequency: 100%|██████████| 7870/7870 [00:38<00:00, 205.35it/s]


In [9]:
#Check if and how many zeros or nulls we got

pairs["pre_FF"].value_counts(dropna=False).sort_index()
pairs["post_FF"].value_counts(dropna=False).sort_index()

post_FF
1      918
2     1358
3     1385
4     1164
5      873
6      671
7      430
8      328
9      220
10     144
11     140
12      80
13      37
14      31
15      20
16       7
17      11
18       3
19       7
20       5
21       9
22       2
23       3
25       3
26       3
27       4
29       3
32       1
33       1
34       2
37       1
39       6
Name: count, dtype: int64

In [10]:
#Columns to drop and keep "Pairs" organized and neat

pairs = pairs.drop(columns=[
    #"fire",
    #"pre_geom_crs",
    #"post_geom_crs",
    "pre_buff_10",
    "post_buff_10",
    #"inside_raster"
], errors="ignore")

In [11]:
# Convert polygon geometry columns to WKT strings (passing the function we created before)

geom_cols = ["pre_geom", "post_geom"] 
for col in geom_cols: to_wkt_column(pairs, col)


In [12]:
# Save updated GeoDataFrame
output_path = r"GEDI_Pairs/GEDI_Footprint_Pairs_Fire_FF.gpkg"
pairs.to_file(output_path, driver="GPKG")

print("🔥 Fire Frequency extraction complete!")
print("Saved to:", output_path)

🔥 Fire Frequency extraction complete!
Saved to: GEDI_Pairs/GEDI_Footprint_Pairs_Fire_FF.gpkg
